# RandomForest Cirrhosis Optimized
Target: Cirrhosis_Status

In [13]:
!pip install -q xgboost

In [36]:
# ==========================================================
# Liver Cirrhosis Prediction using Optimized Random Forest
# ==========================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.ensemble import RandomForestClassifier



# ==========================================================
# Load Dataset
# ==========================================================

df = pd.read_csv("/content/drive/MyDrive/cirrhosis.csv")

df.columns = df.columns.str.strip()


# Remove missing target

df = df.dropna(subset=["Stage"])


print("="*60)
print("Dataset Shape :", df.shape)
print("="*60)



# ==========================================================
# Features / Target
# ==========================================================

X = df.drop(
    columns=["ID","Stage"],
    errors="ignore"
)


y = df["Stage"].astype(int)



print("\nClasses:")
print(y.unique())



# ==========================================================
# Feature Types
# ==========================================================

num_cols = X.select_dtypes(
    include=["int64","float64"]
).columns


cat_cols = X.select_dtypes(
    include=["object"]
).columns



print("\nNumerical Features :",len(num_cols))
print("Categorical Features :",len(cat_cols))



# ==========================================================
# Preprocessing
# ==========================================================

numeric_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    )

])


categorical_pipeline = Pipeline([

    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),

    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )

])



preprocessor = ColumnTransformer([

    (
        "num",
        numeric_pipeline,
        num_cols
    ),

    (
        "cat",
        categorical_pipeline,
        cat_cols
    )

])



# ==========================================================
# Train Test Split
# ==========================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.2,

    random_state=42,

    stratify=y

)



# ==========================================================
# Random Forest Optimized
# ==========================================================

rf = RandomForestClassifier(

    n_estimators=500,

    max_depth=8,

    min_samples_split=5,

    min_samples_leaf=2,

    max_features="sqrt",

    class_weight="balanced",

    bootstrap=True,

    random_state=42,

    n_jobs=-1

)



# ==========================================================
# Pipeline
# ==========================================================

model = Pipeline([

    (
        "preprocessor",
        preprocessor
    ),

    (
        "random_forest",
        rf
    )

])



# ==========================================================
# Training
# ==========================================================

print("\nTraining Random Forest...\n")


model.fit(

    X_train,

    y_train

)



# ==========================================================
# Prediction
# ==========================================================

train_pred = model.predict(X_train)

test_pred = model.predict(X_test)



# ==========================================================
# Evaluation
# ==========================================================

print("="*60)


print(
    "Train Accuracy :",
    round(
        accuracy_score(
            y_train,
            train_pred
        )*100,
        2
    ),
    "%"
)


print(
    "Test Accuracy :",
    round(
        accuracy_score(
            y_test,
            test_pred
        )*100,
        2
    ),
    "%"
)


print("="*60)



print("\nClassification Report\n")


print(
    classification_report(
        y_test,
        test_pred
    )
)



print("\nConfusion Matrix\n")


print(
    confusion_matrix(
        y_test,
        test_pred
    )
)



# ==========================================================
# Cross Validation
# ==========================================================

cv = StratifiedKFold(

    n_splits=5,

    shuffle=True,

    random_state=42

)


scores = cross_val_score(

    model,

    X,

    y,

    cv=cv,

    scoring="accuracy",

    n_jobs=-1

)


print("\nCross Validation Scores")

print(scores)



print(
    "\nMean Accuracy:",
    round(scores.mean()*100,2),
    "%"
)


print(
    "STD:",
    round(scores.std()*100,2),
    "%"
)

Dataset Shape : (412, 20)

Classes:
[4 3 2 1]

Numerical Features : 11
Categorical Features : 7

Training Random Forest...

Train Accuracy : 96.05 %
Test Accuracy : 46.99 %

Classification Report

              precision    recall  f1-score   support

           1       0.00      0.00      0.00         4
           2       0.30      0.37      0.33        19
           3       0.41      0.48      0.44        31
           4       0.74      0.59      0.65        29

    accuracy                           0.47        83
   macro avg       0.36      0.36      0.36        83
weighted avg       0.48      0.47      0.47        83


Confusion Matrix

[[ 0  2  2  0]
 [ 0  7 10  2]
 [ 0 12 15  4]
 [ 0  2 10 17]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Cross Validation Scores
[0.46987952 0.51807229 0.56097561 0.59756098 0.45121951]

Mean Accuracy: 51.95 %
STD: 5.47 %


In [34]:
!pip install lightgbm